# 3.1 Creating Synthetic Data

In [ ]:
#__ Import: Numerical __
import numpy as np

#__ Import: Topophone2 __
from topophone2.modelling import ArrayCovarianceModel
from topophone2.utils.read import convert_to_txt

### 3.1.1 Receiver geometry and frequency vector

In [ ]:
#__ Create receiver geometry __
x = np.array([        # Receiver geometry [m] (receiver, direction [XYZ])
    [0e3, 0e3, 0e3],  #     Receiver #1
    [0e3, 1e3, 0e3],  #     Receiver #2
    [0e3, 0e3, 1e3]   #     Receiver #3
])

#__ Create frequency vector __
frequency = np.linspace(0.5, 4.0, 15)  # Frequency vector [Hz] (frequency,)

### 3.1.2 Signal content

In [ ]:
#__ Signal components __
sigs = [
    'point',               # Plane wave
    'normal',              # Gaussian density-function field
    dict(
        sig = 'isonoise',  # Isotropic noise field
        dim = 2            # ... in horizontal plane
    )
]

E.g. parameters for specification above...

| Signal component | Parameter set |
|---|---|
|`'point'` | $\left(\xi_1, \xi_2, \xi_3\right)$ |
|`'normal'` | $\left(\xi_1, \xi_2, \xi_3, \sigma_1, \sigma_2, \sigma_3, \rho_{12}, \rho_{13}, \rho_{23}\right)$ |
|`'isonoise'` | $\left(s_\mathrm{app}\right)$ |

Each component also has a weight parameter appended to the beginning of the parameter set.

In [ ]:
#__ True parameters __
m = np.array([[
    1.0, -2.5e-3, -1.4e-3, -0.5e-3,                                          # 'point'
    2.0, 0.9e-3,  2.5e-3,  -1.2e-3, 0.3e-3, 0.2e-3, 0.2e-3, 0.4, 0.0, -0.2,  # 'normal'
    0.3, 2.9e-3                                                              # 'isonoise'
]])

### 3.1.3 Synthesise signal

In [ ]:
#__ Forward modeller __
array_covariance_model = ArrayCovarianceModel(
    x    = x,
    freq = 2 * np.pi * frequency,
    sigs = sigs,
    calc_noise_only = True
)

#__ Run forward model (model, frequency, receiver pair) __
Gamma_flat, _, Gamma_noise_flat, _ = array_covariance_model.forward(m)

#__ Expand axes (frequency, receiver, receiver) __
Gamma, _       = array_covariance_model.expand(Gamma_flat[0], None)
Gamma_noise, _ = array_covariance_model.expand(Gamma_noise_flat[0], None)

### 3.1.4 Save to Topophone2 input file

In [ ]:
#__ Pretend number of observations __
N       = 16
N_noise = 32

#__ Write to file __
convert_to_txt(
    frequency   = frequency,
    x           = x,
    Gamma_obs   = Gamma,
    output_path = '../data/eg3_full',
    N_obs       = N
)

#__ Write to file (noise-only) __
convert_to_txt(
    frequency   = frequency,
    x           = x,
    Gamma_obs   = Gamma_noise,
    output_path = '../data/eg3_noise',
    N_obs       = N_noise
)